# Fetal Distress Detection from CTG Signals
### CTU-CHB Intrapartum Cardiotocography Database

This notebook implements a complete end-to-end pipeline for detecting fetal distress
during labour using cardiotocography (CTG) recordings. It covers data loading,
exploration, feature extraction, model training, and evaluation.

**Author:** Devanshu Dhoble  
**Assignment:** Janitri — Intrapartum CTG Analysis

## Section 1: Setup & Imports

In [1]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import wfdb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, accuracy_score,
    precision_score, recall_score, f1_score
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)

# Import custom model module
import sys
sys.path.insert(0, os.path.abspath('.'))
from model import FetalDistressModel, extract_features, get_feature_names

# ---- Configuration ----
DATA_DIR = os.environ.get('CTG_DATA_DIR', r'C:\Users\devan\Downloads\ctu-chb-intrapartum-cardiotocography-database-1.0.0\ctu-chb-intrapartum-cardiotocography-database-1.0.0')
ARTIFACTS_DIR = os.path.join('.', 'artifacts')
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print(f'Artifacts directory: {ARTIFACTS_DIR}')
print(f'Number of records found: {len(glob.glob(os.path.join(DATA_DIR, "*.hea")))}')


Artifacts directory: .\artifacts
Number of records found: 552


## Section 2: Data Loading & Exploration

We parse the `.hea` header files to extract clinical outcomes and maternal metadata.
The outcome fields (pH, BDecf, Apgar1, Apgar5) are stored as comment lines in the
header, prefixed with `#`.

In [2]:
def parse_hea_file(filepath):
    """Parse a WFDB .hea file and extract clinical outcomes from comment lines."""
    outcomes = {'record_id': os.path.basename(filepath).replace('.hea', '')}
    
    # Mapping of header keys to clean column names
    key_map = {
        'pH': 'pH', 'BDecf': 'BDecf', 'pCO2': 'pCO2', 'BE': 'BE',
        'Apgar1': 'Apgar1', 'Apgar5': 'Apgar5',
        'Gest.': 'Gest_weeks', 'Weight(g)': 'Weight_g', 'Sex': 'Sex',
        'Age': 'Age', 'Gravidity': 'Gravidity', 'Parity': 'Parity',
        'Diabetes': 'Diabetes', 'Hypertension': 'Hypertension',
        'Preeclampsia': 'Preeclampsia', 'Pyrexia': 'Pyrexia',
        'Meconium': 'Meconium', 'Presentation': 'Presentation',
        'Induced': 'Induced', 'I.stage': 'I_stage', 'II.stage': 'II_stage',
        'NoProgress': 'NoProgress', 'Deliv.': 'Deliv_type',
        'Pos.': 'Pos_IIst', 'Sig2Birth': 'Sig2Birth',
        'Rec.': 'Rec_type', 'dbID': 'dbID',
    }
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line.startswith('#'):
                continue
            # Remove leading '#' and extra dashes/spaces
            content = line.lstrip('#').strip()
            if content.startswith('-') or content.startswith('!'):
                continue
            parts = content.split()
            if len(parts) < 2:
                continue
            key = parts[0]
            val = parts[-1]  # Take the last token as the value
            
            if '!NotReadyYet!' in val:
                continue
            
            # Map to clean column name
            col = key_map.get(key, key)
            
            try:
                outcomes[col] = float(val) if '.' in val else int(val)
            except ValueError:
                outcomes[col] = val
    
    return outcomes


# Parse all header files
hea_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.hea')))
data_list = [parse_hea_file(f) for f in hea_files]
df = pd.DataFrame(data_list)

print(f'Loaded {len(df)} clinical records.')
print(f'\nColumns: {list(df.columns)}')
print(f'\n--- Numerical Summary ---')
display(df[['pH', 'BDecf', 'Apgar1', 'Apgar5', 'Gest_weeks', 'Weight_g', 'Age']].describe())

Loaded 552 clinical records.

Columns: ['record_id', 'pH', 'BDecf', 'pCO2', 'BE', 'Apgar1', 'Apgar5', 'NICU', 'Seizures', 'HIE', 'Intubation', 'Main', 'Other', 'Gest_weeks', 'Weight_g', 'Sex', 'Age', 'Gravidity', 'Parity', 'Diabetes', 'Hypertension', 'Preeclampsia', 'Liq.', 'Pyrexia', 'Meconium', 'Presentation', 'Induced', 'I_stage', 'NoProgress', 'CK/KP', 'II_stage', 'Deliv_type', 'dbID', 'Rec_type', 'Pos_IIst', 'Sig2Birth']

--- Numerical Summary ---


,pH,Apgar1,Apgar5,Gest_weeks,Age
count,552.000000,552.000000,552.000000,552.000000,552.000000
mean,7.230054,8.262681,9.068841,39.994565,29.673913
std,0.105039,1.624959,1.085613,1.132737,4.538863
min,6.850000,1.000000,4.000000,37.000000,18.000000
25%,7.170000,8.000000,9.000000,39.000000,27.000000
50%,7.250000,9.000000,9.000000,40.000000,30.000000
75%,7.300000,9.000000,10.000000,41.000000,33.000000
max,7.470000,10.000000,10.000000,43.000000,46.000000


In [3]:
# Visualise outcome distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# pH distribution
ax = axes[0]
sns.histplot(df['pH'].dropna(), kde=True, bins=25, ax=ax, color='steelblue')
ax.axvline(7.20, color='red', linestyle='--', linewidth=2, label='Threshold (7.20)')
ax.set_title('Umbilical Artery pH')
ax.legend()

# Apgar at 1 min
ax = axes[1]
sns.histplot(df['Apgar1'].dropna(), kde=False, bins=10, ax=ax, color='teal')
ax.set_title('Apgar Score at 1 min')

# Apgar at 5 min
ax = axes[2]
sns.histplot(df['Apgar5'].dropna(), kde=False, bins=10, ax=ax, color='coral')
ax.axvline(7, color='red', linestyle='--', linewidth=2, label='Threshold (7)')
ax.set_title('Apgar Score at 5 min')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'clinical_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

# Check for missing values in key columns
print('\n--- Missing Values in Key Columns ---')
for col in ['pH', 'BDecf', 'Apgar1', 'Apgar5']:
    n_missing = df[col].isna().sum() if col in df.columns else len(df)
    print(f'  {col}: {n_missing} missing ({n_missing/len(df)*100:.1f}%)')


--- Missing Values in Key Columns ---
  pH: 0 missing (0.0%)
  BDecf: 0 missing (0.0%)
  Apgar1: 0 missing (0.0%)
  Apgar5: 0 missing (0.0%)


## Section 3: Label Definition

We define the binary target label **distressed** using two clinically recognised thresholds:

- **Umbilical-cord pH < 7.20**: indicates clinically significant metabolic acidosis
  (ACOG, FIGO guidelines). A pH below 7.20 is evidence the baby was short on oxygen
  during labour.
- **5-minute Apgar score < 7**: indicates the newborn needed active resuscitation and
  may have been physiologically compromised.

A recording is labelled **distressed (1)** if **either** condition is met.

### Why the OR rule?
Using OR (rather than AND) captures both biochemical *and* clinical indicators of
distress.  It also produces a larger positive class (~15–20%), which helps the model
learn meaningful patterns from a modest 552-recording dataset.

In [4]:
# Drop records that are completely missing both pH and Apgar5
df_clean = df.dropna(subset=['pH', 'Apgar5'], how='all').copy()

# Define the distress label
# For records missing one of the two fields, use only the available field
ph_flag = df_clean['pH'].fillna(999) < 7.20      # NaN pH → not triggered
apgar_flag = df_clean['Apgar5'].fillna(999) < 7   # NaN Apgar → not triggered
df_clean['distressed'] = (ph_flag | apgar_flag).astype(int)

counts = df_clean['distressed'].value_counts()
pcts = df_clean['distressed'].value_counts(normalize=True) * 100

print(f'Total usable records: {len(df_clean)}')
print(f'\nClass Distribution:')
print(f'  Normal (0):     {counts.get(0, 0):>4}  ({pcts.get(0, 0):.1f}%)')
print(f'  Distressed (1): {counts.get(1, 0):>4}  ({pcts.get(1, 0):.1f}%)')

# Plot class balance
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df_clean, x='distressed', palette=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_title('Class Balance')
ax.set_xticklabels(['Normal (0)', 'Distressed (1)'])
ax.set_ylabel('Count')
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'class_balance.png'), dpi=150, bbox_inches='tight')
plt.show()

Total usable records: 552

Class Distribution:
  Normal (0):      370  (67.0%)
  Distressed (1):  182  (33.0%)


## Section 4: Signal Preprocessing & Feature Extraction

For each recording we:
1. Load the physical (scaled) FHR and UC signals from the `.dat` file using `wfdb`.
2. Pass them to `extract_features()` from `model.py`, which:
   - Takes the **last 30 minutes** of the recording.
   - Cleans out-of-range FHR values (< 50 or > 250 bpm).
   - Computes **29 hand-crafted features** (HRV, contraction stats, FHR–UC coupling, etc.).
3. Build the feature matrix `X` and label vector `y`.

In [5]:
features_list = []
valid_labels = []
valid_records = []
failed_records = []

print('Extracting features from all records...')
for i, (idx, row) in enumerate(df_clean.iterrows()):
    record_id = row['record_id']
    record_path = os.path.join(DATA_DIR, record_id)
    try:
        record = wfdb.rdrecord(record_path)
        fhr = record.p_signal[:, 0]  # Channel 0: Fetal Heart Rate
        uc  = record.p_signal[:, 1]  # Channel 1: Uterine Contractions
        
        feats = extract_features(fhr, uc)
        assert len(feats) == 29, f'Expected 29 features, got {len(feats)}'
        
        features_list.append(feats)
        valid_labels.append(row['distressed'])
        valid_records.append(record_id)
    except Exception as e:
        failed_records.append((record_id, str(e)))
    
    if (i + 1) % 100 == 0:
        print(f'  Processed {i + 1}/{len(df_clean)} records...')

X = np.array(features_list)
y = np.array(valid_labels)

print(f'\nDone! Successfully processed {X.shape[0]} records.')
print(f'Feature matrix shape: {X.shape}')
print(f'Label vector shape:   {y.shape}')
if failed_records:
    print(f'\nFailed records ({len(failed_records)}):')
    for rid, err in failed_records[:5]:
        print(f'  {rid}: {err}')

# Show feature summary
feature_names = get_feature_names()
df_feats = pd.DataFrame(X, columns=feature_names)
print('\n--- Feature Summary ---')
display(df_feats.describe().T)

Extracting features from all records...


  Processed 100/552 records...


  Processed 200/552 records...


  Processed 300/552 records...


  Processed 400/552 records...


  Processed 500/552 records...



Done! Successfully processed 552 records.
Feature matrix shape: (552, 29)
Label vector shape:   (552,)

--- Feature Summary ---


,count,mean,std,min,25%,50%,75%,max
fhr_mean,552.0,130.985371,13.250990,94.763527,122.008323,130.482363,139.598664,180.164591
fhr_std,552.0,19.286327,6.384015,1.782542,15.110370,19.225580,23.503861,41.127649
fhr_median,552.0,134.166893,14.217174,93.500000,124.000000,134.000000,143.500000,182.500000
fhr_min,552.0,67.788496,17.797140,50.250000,54.500000,63.750000,75.312500,172.750000
fhr_max,552.0,185.572011,24.088273,129.000000,167.500000,182.000000,198.000000,243.000000
fhr_range,552.0,117.783514,31.970282,6.750000,100.625000,116.000000,137.562500,192.000000
fhr_iqr,552.0,22.522418,12.953808,0.250000,12.500000,20.000000,29.000000,87.750000
fhr_skew,552.0,-0.678605,1.036540,-4.159425,-1.286238,-0.652857,-0.083749,3.397229
fhr_kurtosis,552.0,2.330310,3.868927,-1.428100,-0.010428,1.136457,3.426617,30.599444
rmssd,552.0,3.387381,1.446669,0.677846,2.390631,3.217605,4.151917,10.171125


## Section 5: Train/Test Split

We split the data 80/20 with stratification to preserve the class ratio in both sets.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'  Class 0 (Normal):     {(y_train == 0).sum()}')
print(f'  Class 1 (Distressed): {(y_train == 1).sum()}')
print(f'\nTest set: {X_test.shape[0]} samples')
print(f'  Class 0 (Normal):     {(y_test == 0).sum()}')
print(f'  Class 1 (Distressed): {(y_test == 1).sum()}')

# Save processed data for reproducibility
np.savez(os.path.join(ARTIFACTS_DIR, 'train_data.npz'), X=X_train, y=y_train)
np.savez(os.path.join(ARTIFACTS_DIR, 'test_data.npz'), X=X_test, y=y_test)
np.savez(os.path.join(ARTIFACTS_DIR, 'sample_inference.npz'), X=X_test[:5], y=y_test[:5])
print('\nDatasets saved to artifacts/ directory.')

Training set: 441 samples
  Class 0 (Normal):     296
  Class 1 (Distressed): 145

Test set: 111 samples
  Class 0 (Normal):     74
  Class 1 (Distressed): 37

Datasets saved to artifacts/ directory.


## Section 6: Model Training

We use a `RandomForestClassifier` with:
- 300 trees (good variance reduction)
- Max depth 12 (prevents overfitting on small data)
- `class_weight='balanced'` (automatically up-weights the minority class)

In [7]:
mdl = FetalDistressModel(
    n_estimators=300,
    max_depth=12,
    class_weight='balanced',
    random_state=42
)

print('Training the model...')
mdl.fit(X_train, y_train)

# Save model and scaler
mdl.save(ARTIFACTS_DIR)
print(f'Model trained and saved to {ARTIFACTS_DIR}')
print(f'  - model.pkl  ({os.path.getsize(os.path.join(ARTIFACTS_DIR, "model.pkl")) / 1024:.0f} KB)')
print(f'  - scaler.pkl ({os.path.getsize(os.path.join(ARTIFACTS_DIR, "scaler.pkl")) / 1024:.0f} KB)')

Training the model...


Model trained and saved to .\artifacts
  - model.pkl  (3354 KB)
  - scaler.pkl (1 KB)


## Section 7: Evaluation

We evaluate the trained model on the held-out test set using multiple metrics:
- **ROC-AUC** (primary): threshold-independent discrimination ability
- **Recall / Sensitivity**: fraction of truly distressed cases caught
- **Precision**: fraction of predicted-positive cases that are truly distressed
- **F1-score**: harmonic mean of precision and recall
- **Specificity**: fraction of healthy cases correctly identified

In [8]:
# Generate predictions
y_pred = mdl.predict(X_test)
y_pred_proba = mdl.predict_proba(X_test)  # Returns P(distressed) directly

# Compute metrics
acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
spec = tn / (tn + fp) if (tn + fp) > 0 else 0

print('=' * 40)
print('    EVALUATION RESULTS')
print('=' * 40)
print(f'  Accuracy:    {acc:.4f}')
print(f'  ROC-AUC:     {roc_auc:.4f}')
print(f'  Precision:   {prec:.4f}')
print(f'  Recall:      {rec:.4f}')
print(f'  F1 Score:    {f1:.4f}')
print(f'  Specificity: {spec:.4f}')
print('=' * 40)
print(f'\n  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print(f'\n--- Classification Report ---')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Distressed'], zero_division=0))

    EVALUATION RESULTS
  Accuracy:    0.7027
  ROC-AUC:     0.7330
  Precision:   0.5909
  Recall:      0.3514
  F1 Score:    0.4407
  Specificity: 0.8784

  TP=13  FP=9  FN=24  TN=65

--- Classification Report ---
              precision    recall  f1-score   support

      Normal       0.73      0.88      0.80        74
  Distressed       0.59      0.35      0.44        37

    accuracy                           0.70       111
   macro avg       0.66      0.61      0.62       111
weighted avg       0.68      0.70      0.68       111



In [9]:
# --- Plot 1: Confusion Matrix ---
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Distressed'],
            yticklabels=['Normal', 'Distressed'], ax=ax)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- Plot 2: ROC Curve ---
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('Receiver Operating Characteristic (ROC) Curve')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'roc_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- Plot 3: Precision-Recall Curve ---
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_proba)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall_vals, precision_vals, color='purple', linewidth=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'pr_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- Plot 4: Feature Importances (Top 15) ---
importances = mdl.feature_importances
indices = np.argsort(importances)[-15:]
top_names = [feature_names[i] for i in indices]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(len(indices)), importances[indices], align='center', color='steelblue')
ax.set_yticks(range(len(indices)))
ax.set_yticklabels(top_names)
ax.set_xlabel('Relative Importance')
ax.set_title('Top 15 Feature Importances (Random Forest)')
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'feature_importances.png'), dpi=150, bbox_inches='tight')
plt.show()

print('All evaluation plots saved to artifacts/ directory.')

All evaluation plots saved to artifacts/ directory.


## Section 8: Summary & Observations

### Key Findings

1. **Feature Importance**: Features derived from FHR morphology (baseline heart rate,
   variability metrics, decelerations) and their relationship to uterine contractions
   are the strongest predictors of fetal distress.

2. **Model Performance**: The Random Forest classifier with balanced class weights
   achieves a reasonable baseline. The `class_weight='balanced'` parameter helps
   prioritise recall (catching true distress cases), which is critical in a
   safety-sensitive clinical context.

3. **Clinical Relevance**: The hand-crafted features align with how clinicians
   interpret CTG traces — they look at baseline heart rate, variability, presence
   of decelerations, and the response to contractions.

### Limitations

1. **Small dataset**: 552 recordings is modest for machine learning. Results may
   not generalise to other populations.
2. **Class imbalance**: The minority class is small; further techniques like SMOTE
   or threshold optimisation could be explored.
3. **Retrospective labels**: pH and Apgar are measured at delivery, not during
   labour. The model learns associations, not real-time causal patterns.
4. **Signal quality**: Some recordings have significant missing data in the FHR
   channel, which affects feature reliability.

### Next Steps

- Hyperparameter tuning with cross-validation
- Try gradient-boosted trees (XGBoost, LightGBM)
- Experiment with a 1-D CNN or LSTM on raw signals
- Sliding-window inference for real-time risk monitoring
- External validation on a different hospital's dataset